#Transform Circuits Data
1. Read bronze circuits table
1. Keep only the columns required for analytics(Drop url column)
1. Standarize column names using snake_case(circuitId-> circuit_id, circuitName-> circuit_name)coloumn to make them 
1. Rename coloumns to make the  more meaningful (lat->latitude, long-> longitude)
1. Filter out rows where circuits_id is null(business key validation)
1. Remove dublicates records
1. Transform values of coloumns circuits_name and locality to Title Case
1. Write the trandformed data to silver circuits table

In [0]:
%run ../00-common/01.environment-config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.circuits"
silver_table = f"{catalog_name}.{silver_schema}.circuits"
#

**Step 1 - Read bronze circuits table**

In [0]:
 circuits_df = spark.read.option('versionAsof', 0).table(bronze_table)

In [0]:
circuits_df = spark.table(bronze_table)

In [0]:
display(circuits_df)

**Step 2 - Keep only the columns required for analytics(Drop url column)**

In [0]:
#circuits_selected_df= circuits_df.select(
#    "circuitId",
#   "circuitName",
#    "lat",
#    "long",
#    "locality",
#    "country",
#    "ingenstion_timestamp",
#    "source_file"
#)


In [0]:
from pyspark.sql import functions as f

In [0]:
circuits_selected_df= circuits_df.select(
    f.col("circuitId"),
    f.col("circuitName"),
    f.col("lat"),
    f.col("long"),
    f.col("locality"),
    f.col("country").alias("country_name"),
    f.col("ingenstion_timestamp"),
    f.col("source_file")
)


**Step 3 & 4 Standarize column names**
- Standarize column names using snake_case(circuitId-> circuit_id, circuitName-> circuit_name)
- Rename coloumns to make them more meaningful (lat->latitude, long-> longitude)

In [0]:
circuits_renamed_df =(
    circuits_selected_df                                            .withColumnRenamed("circuitId", "circuit_id")
    .withColumnRenamed("circuitName=", "circuit_name")
    .withColumnRenamed("lat", "latitude")      
    .withColumnRenamed("long", "longitude")
)

In [0]:
circuits_renamed_df = (
    circuits_selected_df
        .withColumnsRenamed({
            "circuitId": "circuit_id",
            "circuitName": "circuit_name",
            "lat": "latitude",
            "long": "longitude"
        })
)

In [0]:
display(circuits_renamed_df)

**Step 5 - Filter out rows where circuits_id is null(business key validation)**

In [0]:
#circuits_valid_df = circuits_renamed_df.filter(
#    "circuit_id is Not Null"
#)

In [0]:
circuits_valid_df = circuits_renamed_df.filter(
    f.col("circuit_id").isNotNull()
)

In [0]:
display(circuits_valid_df)


**Step 6 Remove dublicates records**

In [0]:
circuits_distinct_df = circuits_valid_df.distinct()

In [0]:
circuits_distinct_df = circuits_valid_df.dropDuplicates(["circuit_id"])

In [0]:
display(circuits_distinct_df)

**Step 7 - Transform values of coloumns circuits_name and locality to Title Case**

In [0]:
circuits_final_df = (
    circuits_distinct_df
    .withColumn("circuit_name", f.initcap(f.col("circuit_name")))
    .withColumn("locality", f.initcap(f.col("locality")))
)

In [0]:
display(circuits_final_df)

**Step 8 -  Write the trandformed data to silver circuits table**

In [0]:
(
    circuits_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))